In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [6]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.AGD(
    model=model,
    lr=1,
    radius=1000,
    fletcher=False,
    c1=1e-4,
    tau=0.5,
    line_search_method="backtrack",
    line_search_cond="armijo",
    batch_size=1000
    # batch_size=None
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.0701504498720169
epoch:  1, loss: 0.029338186606764793
epoch:  2, loss: 0.019423063844442368
epoch:  3, loss: 0.018465237691998482
epoch:  4, loss: 0.009361626580357552
epoch:  5, loss: 0.008868793956935406
epoch:  6, loss: 0.006988511420786381
epoch:  7, loss: 0.005493342876434326
epoch:  8, loss: 0.004862339235842228
epoch:  9, loss: 0.004540256690233946
epoch:  10, loss: 0.0042547970078885555
epoch:  11, loss: 0.003345123492181301
epoch:  12, loss: 0.0032340281177312136
epoch:  13, loss: 0.002886864123865962
epoch:  14, loss: 0.002295308280736208
epoch:  15, loss: 0.0020533669739961624
epoch:  16, loss: 0.002013516379520297
epoch:  17, loss: 0.0013129526050761342
epoch:  18, loss: 0.001171654905192554
epoch:  19, loss: 0.0011464939452707767
epoch:  20, loss: 0.0008552712970413268
epoch:  21, loss: 0.0007614597561769187
epoch:  22, loss: 0.0007170799071900547
epoch:  23, loss: 0.0006079300073906779
epoch:  24, loss: 0.0005500391125679016
epoch:  25, loss: 0.0004582

In [7]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9984163851164876
Test metrics:  R2 = 0.9931766318757714


In [8]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.AGD(
    model=model,
    lr=1,
    mu=0.001,
    mu_dec=0.1,
    fletcher=True,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    solver="pinv",
    batch_size=100
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0

/home/eugeniolr/Documents/Eugenio/torch_numopt/src/torch_numopt/utils.py:44: UserWarning: torch.linalg.svd: During SVD computation with the selected cusolver driver, batches 0 failed to converge. A more accurate method will be used to compute the SVD as a fallback. Check doc at https://pytorch.org/docs/stable/generated/torch.linalg.svd.html (Triggered internally at /pytorch/aten/src/ATen/native/cuda/linalg/BatchLinearAlgebraLib.cpp:690.)
  U, S, Vt = torch.linalg.svd(mat)


, loss: 0.1373392790555954
epoch:  1, loss: 0.08889491856098175
epoch:  2, loss: 0.06322991847991943
epoch:  3, loss: 0.018866294994950294
epoch:  4, loss: 0.0135482894256711
epoch:  5, loss: 0.010162226855754852
epoch:  6, loss: 0.009356423281133175
epoch:  7, loss: 0.008811403065919876
epoch:  8, loss: 0.008420211263000965
epoch:  9, loss: 0.008110232651233673
epoch:  10, loss: 0.007866086438298225
epoch:  11, loss: 0.007658427115529776
epoch:  12, loss: 0.0075307246297597885
epoch:  13, loss: 0.006517417263239622
epoch:  14, loss: 0.006221735384315252
epoch:  15, loss: 0.006060482002794743
epoch:  16, loss: 0.005693434271961451
epoch:  17, loss: 0.005494589451700449
epoch:  18, loss: 0.005442919209599495
epoch:  19, loss: 0.0054161762818694115
epoch:  20, loss: 0.00539999408647418
epoch:  21, loss: 0.005389944184571505
epoch:  22, loss: 0.005375780165195465
epoch:  23, loss: 0.005352635867893696
epoch:  24, loss: 0.005338446237146854
epoch:  25, loss: 0.005331188440322876
epoch:  26

In [10]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8111476759510646
Test metrics:  R2 = 0.7561264315918624
